# Concrete QC project - Step 3 (what can be done?)

Goal: turn what we found in Step 2 into tools and actions we can use on site.

| Part | Tool / action | Comes from |
|---|---|---|
| A | 7-day early warning: which pours will probably fail at 28 days | H6 (fly ash slows early strength) |
| B | Predict the 28-day strength from the 7-day result (simple model) | H6 |
| C | Control charts to catch a drop in strength early | H4 (30/20 and 40/20 got weaker) |
| D | Mix review: which mixes are at risk and which have room to spare | H1, H4 |
| E | Slump range review | H2, H3 |
| F | Our recommendations | everything |

### Step 1: Load the clean files

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

project = Path.cwd().parent   # the notebook runs from the notebooks folder, so the project is one level up
processed = project / "data" / "processed"
figures = project / "reports" / "figures"
reports = project / "reports"

cubes = pd.read_csv(processed / "cubes_clean.csv")
mixes = pd.read_csv(processed / "mix_designs.csv")
cubes["date_cast"] = pd.to_datetime(cubes["date_cast"])

day7 = cubes[cubes["age_group"] == "7 days"]
day28 = cubes[cubes["age_group"] == "28 days"]

mix_order = ["AFR-15/20", "AFR-20/20", "AFR-25/20", "AFR-30/20", "AFR-35/20", "AFR-40/20",
             "AFR-W30/20", "AFR-W40/20", "AFR-W50/20", "AFR-W50/14", "AFR-W50/20HS", "AFR-35/28CRCP"]

### Step 2: Match the 7-day and 28-day result of each pour

Same as Step 2 (H6): one row per pour with its 7-day and its 28-day strength.

In [ ]:
key = ["lr", "element", "date_cast", "mix_id"]
seven = day7.groupby(key).agg(strength_7=("strength", "mean"), slump=("measured_slump", "mean"), density=("density", "mean"))
twenty_eight = day28.groupby(key)["strength"].mean().rename("strength_28")

pours = pd.concat([seven, twenty_eight], axis=1).dropna(subset=["strength_7", "strength_28"]).reset_index()
pours = pours.merge(mixes[["mix_id", "strength_class", "fly_ash_percent", "water_binder"]], on="mix_id")
pours["ratio_7_28"] = pours["strength_7"] / pours["strength_28"]
pours["failed_28"] = pours["strength_28"] < pours["strength_class"]

print("Pours with both results:", len(pours))
print("Pours that failed at 28 days:", pours["failed_28"].sum())

## PART A - 7-DAY EARLY WARNING

**Idea:** if a fly ash mix normally reaches 75% of its 28-day strength at 7 days, then to reach the class at 28 days
it needs at least about 75% of the class at 7 days.

**Warning level for each mix = class x the normal 7/28 ratio of that mix**

Example: 30/20 has a ratio of about 0.73 -> warning level = 30 x 0.73 = about 22 MPa at 7 days.
If a 7-day result is below this, raise it with the contractor straight away - do not wait 3 weeks for the 28-day result.

In [ ]:
warning = pours.groupby("mix_id").agg(
    strength_class=("strength_class", "first"),
    normal_ratio=("ratio_7_28", "median"),
    pours=("lr", "count"),
)
warning["warning_level_7day"] = (warning["strength_class"] * warning["normal_ratio"]).round(1)
warning = warning.reindex(mix_order).round(2)
print(warning)

### Step 3: How good is the warning?

We check it against what really happened at 28 days:
- **caught** = warning given AND it failed at 28 days (good)
- **missed** = no warning but it failed (bad)
- **false alarm** = warning given but it passed (annoying, but safe)

In [ ]:
pours["warning"] = pours["strength_7"] < pours["mix_id"].map(warning["warning_level_7day"])

check = pd.crosstab(pours["warning"], pours["failed_28"])
check.index = ["no warning", "warning"]
check.columns = ["passed at 28 days", "failed at 28 days"]
print(check)
print()

caught = ((pours["warning"]) & (pours["failed_28"])).sum()
missed = ((~pours["warning"]) & (pours["failed_28"])).sum()
false_alarm = ((pours["warning"]) & (~pours["failed_28"])).sum()
print("Failures caught at 7 days:", caught, "of", caught + missed, f"({caught / (caught + missed) * 100:.0f}%)")
print("False alarms:", false_alarm, "of", (~pours["failed_28"]).sum(), "pours that passed")

**What this means:** if the warning catches most failures 3 weeks early with only a few false alarms,
it is a simple tool we can use from today - just a table of 7-day warning levels per mix.

In [ ]:
warning.to_csv(reports / "7day_warning_levels.csv")
print("Saved: reports/7day_warning_levels.csv")

## PART B - PREDICT THE 28-DAY STRENGTH (simple model)

The warning in Part A only says pass or fail. Here we try to predict the actual 28-day number.

**Important:** we split the data by POUR (LR number), not by row. Otherwise the same pour could be in the training
data and in the test data, and the model would look better than it really is.

We compare three ways:
1. **Ratio method** - 28-day = 7-day / normal ratio of the mix (no model)
2. **Model 1** - linear regression with only the 7-day strength
3. **Model 2** - 7-day strength + class + fly ash % + water/binder + slump + density

In [ ]:
split = GroupShuffleSplit(test_size=0.25, random_state=42)
train_rows, test_rows = next(split.split(pours, groups=pours["lr"]))
train = pours.iloc[train_rows]
test = pours.iloc[test_rows]
print("Train pours:", len(train), " Test pours:", len(test))

In [ ]:
results = []

# 1. Ratio method (ratio learned on the TRAIN pours only)
ratio = train.groupby("mix_id")["ratio_7_28"].median()
predicted = test["strength_7"] / test["mix_id"].map(ratio)
results.append({"method": "Ratio method", "MAE (MPa)": mean_absolute_error(test["strength_28"], predicted)})

# 2. and 3. Linear regression
models = {
    "Model 1 - 7-day only": ["strength_7"],
    "Model 2 - 7-day + mix + slump + density": ["strength_7", "strength_class", "fly_ash_percent", "water_binder", "slump", "density"],
}
for name, columns in models.items():
    train_ok = train.dropna(subset=columns)
    test_ok = test.dropna(subset=columns)
    model = LinearRegression().fit(train_ok[columns], train_ok["strength_28"])
    predicted = model.predict(test_ok[columns])
    results.append({"method": name,
                    "MAE (MPa)": mean_absolute_error(test_ok["strength_28"], predicted),
                    "R2": model.score(test_ok[columns], test_ok["strength_28"])})

print(pd.DataFrame(results).round(2))

- **MAE** = on average, how many MPa the prediction is off
- **R2** = how much of the variation the model explains (1.0 = perfect)

If the simple ratio method is almost as good as the models, use the ratio method on site - it is easier to explain.

In [ ]:
# Picture: predicted vs real 28-day strength (Model 2)
columns = models["Model 2 - 7-day + mix + slump + density"]
train_ok = train.dropna(subset=columns)
test_ok = test.dropna(subset=columns)
model = LinearRegression().fit(train_ok[columns], train_ok["strength_28"])
predicted = model.predict(test_ok[columns])

plt.figure(figsize=(6, 6))
plt.scatter(test_ok["strength_28"], predicted, s=10, alpha=0.5)
plt.plot([10, 80], [10, 80], color="red", label="perfect prediction")
plt.xlabel("Real 28-day strength (MPa)")
plt.ylabel("Predicted 28-day strength (MPa)")
plt.title("Predicted vs real 28-day strength (test pours)")
plt.legend()
plt.tight_layout()
plt.savefig(figures / "11_prediction_28day.png", dpi=150)
plt.show()

## PART C - CONTROL CHARTS (catch a drop early)

In Step 2 we found that 30/20 and 40/20 got weaker inside the same element.
A control chart would have shown this much earlier.

We plot every 28-day result in date order, plus the **average of the last 10 results**.
If the line of the last 10 drifts down towards the class, it is time to ask the supplier questions.

In [ ]:
def control_chart(mix, file_name):
    one = day28[day28["mix_id"] == mix].dropna(subset=["date_cast"]).sort_values("date_cast")
    class_value = one["strength_class"].iloc[0]
    target = mixes.loc[mixes["mix_id"] == mix, "target"].iloc[0]
    one["last_10_average"] = one["strength"].rolling(10).mean()

    plt.figure(figsize=(11, 4))
    plt.scatter(one["date_cast"], one["strength"], s=8, alpha=0.4, label="28-day result")
    plt.plot(one["date_cast"], one["last_10_average"], color="black", label="average of last 10")
    plt.axhline(target, color="green", linestyle="--", label=f"design target {target}")
    plt.axhline(class_value, color="red", label=f"class {class_value}")
    plt.ylabel("28-day strength (MPa)")
    plt.title(f"Control chart - {mix}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(figures / file_name, dpi=150)
    plt.show()

control_chart("AFR-30/20", "12_control_chart_30MPa.png")
control_chart("AFR-40/20", "13_control_chart_40MPa.png")
control_chart("AFR-W40/20", "14_control_chart_W40MPa.png")

**Rule we can use:** if the average of the last 10 results drops below the design target, ask the supplier for
their own plant results and recent cement / fly ash certificates. W40/20 is shown as a "good" example to compare.

## PART D - MIX REVIEW

For each mix we work out the **characteristic strength** = mean - 1.64 x standard deviation.
This is the strength that 95% of results should reach. It should be at or above the class.

- If it is **below the class** -> too many low results, the mix or its control needs attention
- If the mean is **far above the design target** -> the mix may have room to save cement (only with the Engineer and supplier)

In [ ]:
review = day28.groupby("mix_id")["strength"].agg(["count", "mean", "std"])
review["characteristic"] = review["mean"] - 1.64 * review["std"]
review = review.join(mixes.set_index("mix_id")[["strength_class", "target", "cement", "fly_ash"]])
review["fail_percent"] = day28.groupby("mix_id")["passed"].apply(lambda x: (x == False).mean() * 100)

def decide(row):
    if row["count"] < 30:
        return "too few results to judge"
    if row["characteristic"] < row["strength_class"]:
        return "AT RISK - too many low results"
    if row["mean"] > row["target"] * 1.15:
        return "room to optimise"
    return "OK"

review["action"] = review.apply(decide, axis=1)
review = review.reindex(mix_order).round(1)
print(review[["count", "strength_class", "target", "mean", "std", "characteristic", "fail_percent", "action"]])

review.to_csv(reports / "mix_review.csv")
print()
print("Saved: reports/mix_review.csv")

## PART E - SLUMP RANGE REVIEW

In Step 2 we saw that concrete above the design slump was NOT weaker (H2), and that 30/20 is too wet in every
element (H3). So the design range may simply not match how the concrete is really supplied and placed.

Here we look at the slump range that most results really have (the middle 80%: from the 10th to the 90th percentile).

In [ ]:
slump = cubes.groupby("mix_id")["measured_slump"].describe(percentiles=[0.1, 0.5, 0.9])[["count", "10%", "50%", "90%"]]
slump = slump.join(mixes.set_index("mix_id")[["design_slump_min", "design_slump_max"]])
slump["above_design_percent"] = cubes.groupby("mix_id")["slump_check"].apply(lambda x: (x == "above").mean() * 100)
print(slump.reindex(mix_order).round(0))

**What to do with it:** this does NOT mean we just change the spec. It is evidence to discuss with the Engineer
and the supplier: either the supplier sends the concrete inside the range, or the range on the D2 is formally
revised (with trial results to show strength and durability are not affected).

## PART F - OUR RECOMMENDATIONS

Write them in your own words. Suggested structure - one line per finding:

1. **7-day early warning** - use the warning levels table on every 7-day result (Part A).
2. **Strength drop in 30/20 and 40/20** - start control charts and ask the supplier about cement / fly ash changes (Part C, H4).
3. **Low-strength mixes (15/20, 20/20, 35/20)** - erratic in every element; check sampling and curing on small jobs, and make sure the report gives the full mix (stone size) (Part D, H1).
4. **Slump** - review the design range with the supplier and the Engineer instead of rejecting loads (Part E, H2, H3).
5. **Durability** - all tests pass with margin; keep testing, but OPI and sorptivity can not be predicted from cube strength alone (H7).
6. **Data quality** - ask the lab about the copied chloride values, and keep the report fields complete (stone size, slump, supplier).

### Our recommendations

1. **Use the 7-day early warning on every result.** Check each 7-day result against the warning level for its mix (`reports/7day_warning_levels.csv`, e.g. 30/20 below 21.8 MPa, W40/20 below 30.0 MPa). On past data this caught 86% of 28-day failures three weeks early with few false alarms. A result below the level is raised with the contractor and supplier straight away, not after the 28-day result.

2. **Investigate the strength drop in 30/20 and 40/20.** Both got weaker inside the same elements (30/20 MSE walls 44 -> about 40 MPa, 40/20 63 -> about 48-50 MPa). The 7-day strength dropped too, while slump and density stayed the same, which points to the binder (cement / fly ash) rather than site practice. Ask AfriSam for their plant results and recent cement and fly ash certificates for 2025-2026, and keep a control chart (average of the last 10 results) for these two mixes.

3. **Tighten control of the low-strength mixes (15/20, 20/20, 25/20, 35/20).** Their results swing a lot (cov 23-46%) in every element, and 20/20 fails 23% of the time. Check how cubes are sampled, made and cured on small jobs (stormwater, duct encasement), where there is often less supervision. Make sure every report gives the full mix, class and stone size (e.g. 20MPa/20) so results can be matched to the right D2.

4. **Review the design slump range instead of rejecting loads.** Concrete above the 75-125 mm range was not weaker or lighter, so the extra slump is most likely admixture, not site water. 30/20 is too wet in all elements, 40/20 mainly in parapets (77%). Discuss with the Engineer and AfriSam: either the supply stays inside the D2 range, or the range is formally revised with trial results showing strength and durability are not affected.

5. **Keep durability testing as it is.** All 198 tests pass with margin (OPI 10.3 vs 9.3, sorptivity 6.7 vs 10) and results have improved since 2023. Cube strength is only weakly linked to durability, so it can not replace the durability tests. Parapets, pile caps and the WB retaining wall have the smallest margin and should be watched first.

6. **Fix the data quality issues with the lab.** Query SWL about the 115 reports with identical chloride values (0.299 / 0.377 / 0.426 / 0.288). Correct errors such as impossible strengths (e.g. 272.5 MPa), the negative sorptivity and porosity on LR-3633/3634, and wrong cast dates. Ask for complete report fields: supplier, mix, stone size and measured slump.

7. **Optional (with the Engineer and supplier): look at cement optimisation.** 40/20 and the CRCP run well above their design targets. Once the drop in point 2 is resolved, there may be room to reduce cement (cost and CO2), but only through trial mixes and approval, not from this data alone.